# Entity Archetype Explorer

**Schema(s)** defining this data:
- `contracts/schemas/archetypes.schema.json` — **ArchetypesPack**: `archetypes[]` with `archetypeId`, `label`, `description`, `vectorProfile`, `featureProfile`, `preferredSkills`, `visual`. Narrative stat names (in profiles) from STAT-AND-BEHAVIOUR-TAXONOMY (do not call them "features"; room feature = room type only).

**Asset / pack** we load:
- `contracts/data/content_archetypes.json` — the archetype pack (view, rebalance profile values and preferredSkills here or in the app).

**What this tool does:** View the pack; adjust **narrative stats** (the single 15-name set) with sliders to see how classification changes; rebalance archetype profiles (vectorProfile / featureProfile) and preferredSkills to tune behaviour. Same classification logic as the engine (`ArchetypeDirector.rank` / `classify`).

*Run cells in order. Interactive app requires `ipywidgets` (`pip install ipywidgets`, then restart kernel).*

In [ ]:
import json
from pathlib import Path

# Repo root and contracts paths (schema + pack)
ROOT = Path.cwd()
if not (ROOT / "packages" / "engine").is_dir() and (ROOT.parent / "packages" / "engine").is_dir():
    ROOT = ROOT.parent
CONTRACTS_DIR = ROOT / "packages" / "engine" / "src" / "escape-the-dungeon" / "contracts"
SCHEMA_PATH = CONTRACTS_DIR / "schemas" / "archetypes.schema.json"
PACK_PATH = CONTRACTS_DIR / "data" / "content_archetypes.json"

with open(PACK_PATH, encoding="utf-8") as f:
    ARCHETYPE_PACK = json.load(f)

print("Schema:", SCHEMA_PATH.relative_to(ROOT))
print("Pack:  ", PACK_PATH.relative_to(ROOT))
print(f"Loaded {len(ARCHETYPE_PACK['archetypes'])} archetypes")

## View pack

Archetypes in the loaded pack: profile keys and preferred skills. **Rebalance:** edit `content_archetypes.json` (vectorProfile, featureProfile, preferredSkills) and re-run the load cell to see effect on classification.

In [ ]:
# View: compact table of archetypes (profile keys + preferredSkills)
for a in ARCHETYPE_PACK["archetypes"]:
    vp = list((a.get("vectorProfile") or {}).keys())
    fp = list((a.get("featureProfile") or {}).keys())
    skills = a.get("preferredSkills") or []
    print(f"{a['archetypeId']:12} | {a['label']:12} | vectorProfile: {vp} | featureProfile: {fp} | skills: {skills}")

In [ ]:
# Match engine: core/types.ts TRAIT_NAMES and FEATURE_NAMES
TRAIT_NAMES = [
    "Comprehension", "Constraint", "Construction", "Direction", "Empathy",
    "Equilibrium", "Freedom", "Levity", "Projection", "Survival",
]
FEATURE_NAMES = ["Fame", "Effort", "Awareness", "Guile", "Momentum"]


def _vec_from_keys(source: dict, keys: list) -> dict:
    return {k: float(source.get(k, 0) or 0) for k in keys}


def _cosine(a: dict, b: dict, keys: list) -> float:
    dot = sum((a.get(k, 0) or 0) * (b.get(k, 0) or 0) for k in keys)
    mag_a = (sum((a.get(k, 0) or 0) ** 2 for k in keys)) ** 0.5
    mag_b = (sum((b.get(k, 0) or 0) ** 2 for k in keys)) ** 0.5
    if mag_a <= 1e-9 or mag_b <= 1e-9:
        return 0.0
    return max(-1, min(1, dot / (mag_a * mag_b)))


def rank_archetypes(entity_traits: dict, entity_features: dict, entity_skills: dict, archetypes: list) -> list:
    """Same formula as engine ArchetypeDirector.rank: trait cosine * 0.65 + feature cosine * 0.25 + skill bonus (max 0.24)."""
    t = _vec_from_keys(entity_traits, TRAIT_NAMES)
    f = _vec_from_keys(entity_features, FEATURE_NAMES)
    rows = []
    for defn in archetypes:
        tp = _vec_from_keys(defn.get("vectorProfile") or {}, TRAIT_NAMES)
        fp = _vec_from_keys(defn.get("featureProfile") or {}, FEATURE_NAMES)
        trait_score = _cosine(t, tp, TRAIT_NAMES)
        feature_score = _cosine(f, fp, FEATURE_NAMES)
        preferred = defn.get("preferredSkills") or []
        unlocked = sum(1 for sid in preferred if entity_skills.get(sid))
        skill_score = min(0.24, unlocked * 0.08)
        score = trait_score * 0.65 + feature_score * 0.25 + skill_score
        rows.append({"archetypeId": defn["archetypeId"], "label": defn["label"], "score": score})
    return sorted(rows, key=lambda r: r["score"], reverse=True)


def classify(current_heading: str, ranked: list, min_score: float = 0.12, hysteresis: float = 0.05) -> str:
    """Same logic as engine ArchetypeDirector.classify: best wins unless below threshold or hysteresis keeps current."""
    if not ranked:
        return current_heading
    best = ranked[0]
    if best["score"] < min_score:
        return current_heading
    current = next((r for r in ranked if r["archetypeId"] == current_heading), None)
    if current and (best["score"] - current["score"]) < hysteresis:
        return current_heading
    return best["archetypeId"]

In [ ]:
# Default entity: all stats 0, no skills. Edit these dicts and re-run the next cell to see archetype change.
entity_traits = {k: 0.0 for k in TRAIT_NAMES}
entity_features = {k: 0.0 for k in FEATURE_NAMES}
entity_skills = {}  # skillId -> True if unlocked
current_heading = "wanderer"

# Example: bump some stats and see Delver or Warden win
entity_traits["Survival"] = 0.3
entity_traits["Construction"] = 0.2
entity_features["Guile"] = 0.15
entity_features["Awareness"] = 0.1

In [ ]:
ranked = rank_archetypes(entity_traits, entity_features, entity_skills, ARCHETYPE_PACK["archetypes"])
picked = classify(current_heading, ranked)

print("Top archetypes (score)")
for r in ranked[:5]:
    mark = " <-- picked" if r["archetypeId"] == picked else ""
    print(f"  {r['label']}: {r['score']:.3f}{mark}")
print(f"\nClassify result: {picked}")

## Interactive app

Adjust sliders below; **Picked** and **Top archetypes** update live. Requires `ipywidgets` (`pip install ipywidgets`).

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    _have_widgets = True
except ImportError:
    from IPython.display import display, HTML
    display(HTML("<p><b>Install ipywidgets for the app UI:</b> <code>pip install ipywidgets</code>, then restart the kernel.</p><p>Or use the programmatic cell above: set <code>entity_traits</code> / <code>entity_features</code> and re-run the result cell.</p>"))
    _have_widgets = False

if _have_widgets:

    def make_slider(name, value=0):
        return widgets.FloatSlider(value=value, min=0, max=1, step=0.05, description=name[:12], style={"description_width": "100px"})

    trait_sliders = {k: make_slider(k) for k in TRAIT_NAMES}
    feature_sliders = {k: make_slider(k) for k in FEATURE_NAMES}
    current_heading = widgets.Dropdown(options=["wanderer", "delver", "warden", "hunter", "tactician", "showrunner"], value="wanderer", description="Current heading:")

    result_html = widgets.HTML(value="")
    out = widgets.Output()

    def update(_=None):
        traits = {k: s.value for k, s in trait_sliders.items()}
        features = {k: s.value for k, s in feature_sliders.items()}
        ranked = rank_archetypes(traits, features, {}, ARCHETYPE_PACK["archetypes"])
        picked = classify(current_heading.value, ranked)
        rows = "".join(
            f"<tr><td>{r['label']}</td><td>{r['score']:.3f}</td><td>{'✓' if r['archetypeId'] == picked else ''}</td></tr>"
            for r in ranked[:6]
        )
        result_html.value = f"""
    <div style="margin:12px 0; padding:12px; border:1px solid #ccc; border-radius:8px; background:#fafafa;">
      <p style="margin:0 0 8px 0;"><strong>Picked:</strong> <span style="font-size:1.1em;">{picked}</span></p>
      <table style="border-collapse:collapse; width:100%; max-width:280px;">
        <thead><tr><th style="text-align:left;">Archetype</th><th>Score</th><th></th></tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
        """

    for w in list(trait_sliders.values()) + list(feature_sliders.values()) + [current_heading]:
        w.observe(update)
    update()

    traits_box = widgets.VBox([widgets.HTML("<b>Narrative stats (vectorProfile, 10)</b>")] + [widgets.HBox([trait_sliders[k]]) for k in TRAIT_NAMES])
    features_box = widgets.VBox([widgets.HTML("<b>Narrative stats (other 5)</b>")] + [widgets.HBox([feature_sliders[k]]) for k in FEATURE_NAMES])
    display(widgets.HBox([widgets.VBox([current_heading, traits_box]), widgets.VBox([features_box]), widgets.VBox([widgets.HTML("<b>Result</b>"), result_html])]))